### Libraries

In [1]:
import os, json, cv2
import pandas as pd
import mediapipe as mp
import numpy as np
from math import hypot
from tqdm import tqdm

### Initialize MediaPipe FaceMesh

In [2]:
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh(static_image_mode=True, max_num_faces=1)

### EAR Calculation

In [3]:

def get_ear(eye):
    # Vertical distances
    v1 = hypot(eye[1][0]-eye[5][0], eye[1][1]-eye[5][1])
    v2 = hypot(eye[2][0]-eye[4][0], eye[2][1]-eye[4][1])
    # Horizontal distance
    h = hypot(eye[0][0]-eye[3][0], eye[0][1]-eye[3][1])
    return (v1 + v2) / (2.0 * h + 1e-6)

### MAR Calculation

In [4]:
def get_mar(mouth):
    # Vertical distance (Inner lips: 13 and 14)
    v = hypot(mouth[0][0]-mouth[1][0], mouth[0][1]-mouth[1][1])
    # Horizontal distance (Corners: 78 and 308)
    h = hypot(mouth[2][0]-mouth[3][0], mouth[2][1]-mouth[3][1])
    return v / (h + 1e-6)

### Dataset Path

In [5]:

ROOT = "./Dataset/FL3D_Dataset/classification_frames/"
rows = []

### Process Frames

In [7]:
for video in tqdm(os.listdir(ROOT), desc="Processing Videos"):
    vpath = os.path.join(ROOT, video)
    if not os.path.isdir(vpath): continue

    json_files = [f for f in os.listdir(vpath) if f.endswith(".json")]
    if not json_files: continue
    labels = json.load(open(os.path.join(vpath, json_files[0])))

    for frame_name in os.listdir(vpath):
        if not frame_name.endswith(".jpg") or frame_name not in labels: continue

        label = labels[frame_name]["driver_state"]
        img = cv2.imread(os.path.join(vpath, frame_name))
        if img is None: continue

        h, w = img.shape[:2]
        res = face_mesh.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            
            # Left Eye Indices
            le = [(int(lm[i].x*w), int(lm[i].y*h)) for i in [33, 160, 158, 133, 153, 144]]
            # Right Eye Indices
            re = [(int(lm[i].x*w), int(lm[i].y*h)) for i in [362, 385, 387, 263, 373, 380]]
            # Mouth Indices (Vertical: 13,14 | Horizontal: 78,308)
            mo = [(int(lm[i].x*w), int(lm[i].y*h)) for i in [13, 14, 78, 308]]

            ear_val = (get_ear(le) + get_ear(re)) / 2.0
            mar_val = get_mar(mo)

            # Filtering Outliers: Normal MAR values range from 0 to 1. 
            # If it's > 2, it's definitely an error.
            if mar_val < 2.0:
                rows.append([ear_val, mar_val, label])


Processing Videos: 100%|███████████████████| 49/49 [49:00<00:00, 60.01s/it]


### Save CSV

In [ ]:
df = pd.DataFrame(rows, columns=["EAR", "MAR", "Label"])
df.to_csv("./Dataset/MLP_Dataset/MLP_raw_v2.csv", index=False)
print("✅ STEP 1 DONE: Features extracted with correct MAR logic.")